# 01 · Input-pipeline check on a GPU  (issue #9)

Checks the shared input pipeline on the hardware we train on:
1. Rebuilds the processed cache with `scripts/prepare_data.py`. It also **verifies that the frozen `split_v1` and band stats are reproduced exactly** on this machine.
2. Benchmarks DataLoader + GPU preprocessing throughput (target **> 1500 img/s** on a T4).
3. Saves the augmentation-examples figure.

**Kaggle:** *Settings → Accelerator: GPU T4 ×2 (or P100)*, *Settings → Internet: On*, then *Run All* (about 15 min, mostly the download).
**Colab:** *Runtime → Change runtime type → T4 GPU*, then *Run all*.
At the end, copy the printed summary back (and download `augmentation_examples.png` from the output panel).

In [ ]:
BRANCH = "main"
REPO = "https://github.com/ErenKerimoglu/CS464_Project.git"

import os, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
DATA_DIR = f"{WORK}/data"
os.environ["GALAXY_DATA_DIR"] = DATA_DIR
if not os.path.exists(f"{WORK}/CS464_Project"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, f"{WORK}/CS464_Project"], check=True)
%cd {WORK}/CS464_Project
!git log --oneline -1
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2

## 1 · Data (attached `galaxy10-decals-processed-v1` dataset, otherwise download + build)

In [ ]:
import galaxy.constants as C
if C.PROCESSED_IMAGES.exists():
    print('Using attached dataset:', C.PROCESSED_IMAGES)   # team Kaggle dataset → no download needed
else:
    !python scripts/prepare_data.py


## 2 · Throughput benchmark

In [ ]:
import json, subprocess, sys
runs = [
    ("rotflip224_memmap_w4", ["--workers", "4"]),
    ("rotflip224_ram_w2",    ["--workers", "2", "--in-memory"]),
    ("rotflip224_ram_w4",    ["--workers", "4", "--in-memory"]),
    ("none224_ram_w4",       ["--workers", "4", "--in-memory", "--policy", "none"]),
    ("rotflip128_ram_w4",    ["--workers", "4", "--in-memory", "--out-size", "128"]),
]
results = {}
for name, args in runs:
    out = subprocess.run([sys.executable, "scripts/benchmark_pipeline.py", "--batches", "150", *args],
                         capture_output=True, text=True)
    if out.returncode:
        print(name, "FAILED\n", out.stderr[-2000:]); continue
    results[name] = json.loads(out.stdout[out.stdout.index("{"):out.stdout.rindex("}") + 1])
    r = results[name]
    print(f"{name:<22} loader {r['loader_img_s']:>6} | loader+preprocess {r['loader_plus_preprocess_img_s']:>6} "
          f"| preprocess only {r['preprocess_only_img_s']:>6} img/s")

## 3 · Augmentation examples

In [ ]:
!python scripts/benchmark_pipeline.py --workers 0 --batches 3 --figure results/figures/eda/augmentation_examples.png > /dev/null
from IPython.display import Image, display
display(Image("results/figures/eda/augmentation_examples.png"))

## 4 · Summary (paste this back)

In [ ]:
import torch
summary = {"gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu",
           "cpus": os.cpu_count(), "results": results}
print(json.dumps(summary, indent=1))